Q learning, main purpose is to learn value of action, then using action value to determine policy.

Policy is the ultimate goal, but not the **directly** learned object.

---
### REINFORCE intro
> Why not make learning policy the main task?
>
> Can we use sampled rewards to update the policy in the direction of higher expected reward?

our goal:
$$
\max_\theta J(\theta)
$$

how to calculate gradient:

$$
\nabla_\theta J
=
\mathbb E\left[
R\,\nabla_\theta\log\pi_\theta(a|s)
\right]
$$

We  want **gradient ascend**, Pytorch only provides **gradient descend** so we use a **surrogate loss**:

The goal is to minizie:
> loss = -policy.log_prob(action) * reward

We know this will increase the action's probablity (covered in llm series).
> min(-policy.log_prob(action))

the larger the reward, should affect more:
> min(-policy.log_prob(action) * reward)

An example:
```text
dance -> +1
forward -> +10
```

If agent picks dance, it will increase its P (according to equation above). You say:
>but dance is worse!

Yes, if we pick forward, its P will increase more!

REINFORCE doesn't need **every** sample update to be correct. It says: when do it enough time, the expected sample updates is J's gradient.

# REINFORCE (Williams, 1992)

**Core idea:** Optimize a stochastic policy directly using sampled rewards, without first learning a value function.

## Theorem 1 — Immediate Reward

**Problem:** How can we update a policy without knowing or differentiating the environment?

**Example:**

A robot chooses between two buttons:

- Button A → 70% chance of Reward = 1
- Button B → 30% chance of Reward = 1

Initially, the policy assigns 50% probability to each button.

Suppose the robot selects B and happens to receive a reward. REINFORCE increases the probability of choosing B.

**But a single outcome is random. Just because an action received a reward doesn't mean increasing its probability is the correct direction. How can we prove the update rule is reliable?**

REINFORCE update:

$$
\Delta\theta=\alpha R\nabla_\theta\log\pi_\theta(a\mid s)
$$

**Guarantee:** Under the theorem's assumptions, the expected update follows the true reward (a.k.a the expected reward) gradient.

$$
\mathbb E[\Delta\theta]=\alpha\nabla_\theta J(\theta)
$$

## Theorem 2 — Delayed Reward

**Problem:** What if multiple actions are taken before receiving a reward?

**Example:**

The robot must press two buttons in sequence. It receives a reward only after both choices.

- A → A: Reward = 1
- A → B: Reward = 0
- B → A: Reward = 0
- B → B: Reward = 0

The robot receives no feedback after the first action.

**How can the final reward be used to update the policy for both actions?**

Accumulate the log-probability gradients and multiply by the final reward:

$$
\boxed{
\Delta\theta=\alpha R\sum_{t=1}^{T}
\nabla_\theta\log\pi_\theta(a_t\mid s_t)
}
$$

**Why addition?**

- Trajectory probability = product of action probabilities (and environment terms).
- Log turns products into sums.
- Gradient of a sum = sum of gradients.

Under the theorem's episodic assumptions, the expected update still follows the correct gradient.

## Limitations

- **Unbiased ≠ guaranteed convergence.**
- Individual updates can be noisy; variance can be very high.
- A final reward does not tell us which actions were responsible.
- Baselines and value-function estimates can reduce variance, motivating Actor-Critic methods.

**Takeaway:** REINFORCE provides a mathematically justified way to learn a policy directly from sampled rewards, even when the environment is unknown or non-differentiable.

### Issue: high variance during gradient ascend

For example, say each time check gradient is like checking compass. Say you get results: [200, 100, -100, -200], the right direction is 0.

The updates **fluctuate significantly** even though their expectation is zero. High variance makes optimization noisy and can require more samples to make reliable progress.

Lots of back and forth, wasted effort. This will make the journey very inefficient.

How to solve this?

Baseline is a nature idea:

dance -> +1
forward -> +10

why we still need to increase dance?

We can have a baseline, if you are better -> increase, worse -> decrease P.

and subtracting an **action-independent** baseline **does not change** the expected policy gradient.:

$$
\mathbb E[(R-b)\nabla\log\pi]
=
\mathbb E[R\nabla\log\pi]
$$


<br><br>

#### What should be baseline?

Say a state has 3 actions, the baseline should be the **state's expected return** (average action's return in this state).

baseline need to be **action-independent**. Compute Advantage:
$$
A_t=G_t-V(s_t)
$$
using a baseline that has **NOT been updated** with the current sample.
